# cdsep walkthrough

A 5-minute tour of **Control–Data Flow Separation** for multi-agent LLM optimization. We will:

1. Define a typed control schema and an Agent.
2. Run a single-agent episode.
3. Build a leader–worker pipeline and route by typed control.
4. Show that schema validation prevents bad control fields from propagating.
5. Optimize a prompt with the *separated* TextGrad optimizer and watch stability stay at 100 %.

You'll need `OPENAI_API_KEY` set. The notebook uses `gpt-5.4-nano` for fast agent calls and `gpt-5.4-mini` for the optimizer.

In [ ]:
from dataclasses import dataclass
from typing import Literal
from cdsep import Agent, LLMClient, run_single_agent_episode, run_episode

llm = LLMClient(model='gpt-5.4-nano', temperature=0)

## 1. A typed control schema is just a dataclass

Every Agent in cdsep is parameterised by a control schema (a dataclass or Pydantic model). The *types* of the fields determine what the LLM is allowed to emit; `Literal` is especially useful for enumerated values like rating levels or routing targets.

In [ ]:
@dataclass
class Sentiment:
    label: Literal['positive', 'negative', 'neutral']
    confidence: float

agent = Agent('classifier', Sentiment,
    system_prompt='Classify the sentiment of the following text. Provide a confidence in [0, 1].')

trace = run_single_agent_episode(agent, 'I love this library!', llm)
print(trace.steps[0].control)

## 2. Routing on typed control objects

In a multi-agent pipeline, the developer writes a Python `route_fn` that takes the typed control object and returns the next agent's name (or `"terminate"`). Because the control schema is enforced at parse time, the optimizer cannot produce a routing field that names a nonexistent agent.

In [ ]:
@dataclass
class CoordCtrl:
    route_to: Literal['math', 'word', 'done']

@dataclass
class SpecCtrl:
    answer: str

coord = Agent('coord', CoordCtrl,
    system_prompt='Pick "math" for arithmetic, "word" for text questions, or "done" if you have the final answer.')
math = Agent('math', SpecCtrl, 'You are a math specialist.')
word = Agent('word', SpecCtrl, 'You are a text/word analysis specialist.')

def route(c):
    if isinstance(c, CoordCtrl):
        return 'terminate' if c.route_to == 'done' else c.route_to
    return 'coord'

trace = run_episode(coord, {'coord': coord, 'math': math, 'word': word},
                    route, 'How many letters in Mississippi?', llm, max_steps=6)
for s in trace.steps:
    print(f'  {s.agent_name:<6}  {s.control}')
print('Stable:', trace.is_stable)

## 3. Why this is safer than a one-string prompt

Compare with `separated=False` (the naive baseline). The agent has no auto-scaffolding -- format instructions are part of the editable system prompt, and there is no parse-retry safety net. If the optimizer accidentally removes the JSON instructions, parsing fails and the episode crashes.

In [ ]:
from cdsep import parse_response, validate_control

# A garbage response from a misaligned naive prompt
raw = 'I think the answer is 7, but it could be 3.'
ctrl_dict, msg = parse_response(raw)
print('parsed:', ctrl_dict)
ctrl, errors = validate_control(ctrl_dict, Sentiment)
print('errors:', errors)

In our framework (`separated=True`) such a parse failure triggers a repair turn; if all retries are exhausted the agent reports the failure cleanly. Routing logic never sees a malformed object.

## 4. End-to-end optimization

See `examples/03_textgrad_optimization.py` for a complete optimization loop. The TL;DR: with `separated=True`, you can throw arbitrarily aggressive prompt edits at your pipeline without ever breaking the routing surface. Stability stays at 100 % by construction; the only thing that varies is task quality.